In [1]:
import os, random, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from imblearn.metrics import specificity_score
from mambapy.vim import VMamba, MambaConfig
from thop import profile

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
class VimEncoder(nn.Module):
    def __init__(self, d_model=32, n_layers=2, d_state=16):
        super().__init__()
        config = MambaConfig(d_model=d_model, n_layers=n_layers, d_state=d_state,
                              bidirectional=True, divide_output=True, pscan=True, use_cuda=False)
        self.encoder = VMamba(config)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, tokens):
        return self.final_norm(self.encoder(tokens))

In [3]:
class ROIPatchEmbed3D(nn.Module):
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32):
        super().__init__()
        self.n_rois = n_rois
        self.patch_size = patch_size
        self.grid_size = roi_size // patch_size
        self.patches_per_roi = self.grid_size ** 3
        self.d_model = d_model
        self.patch_conv = nn.Conv3d(1, d_model, kernel_size=patch_size, stride=patch_size)
        self.roi_embed = nn.Embedding(n_rois, d_model)
        self.depth_embed = nn.Embedding(self.grid_size, d_model)
        self.height_embed = nn.Embedding(self.grid_size, d_model)
        self.width_embed = nn.Embedding(self.grid_size, d_model)
        with torch.no_grad():
            for emb in [self.roi_embed, self.depth_embed, self.height_embed, self.width_embed]:
                emb.weight.mul_(0.02)
        d, h, w = torch.meshgrid(torch.arange(self.grid_size), torch.arange(self.grid_size),
                                  torch.arange(self.grid_size), indexing="ij")
        self.register_buffer("coordinates", torch.stack([d, h, w], dim=-1).reshape(-1, 3), persistent=False)

    def forward(self, rois):
        batch_size, n_rois = rois.shape[:2]
        x = rois.reshape(batch_size * n_rois, 1, rois.shape[-3], rois.shape[-2], rois.shape[-1])
        tokens = self.patch_conv(x).flatten(2).transpose(1, 2)
        tokens = tokens.reshape(batch_size, n_rois, self.patches_per_roi, self.d_model)
        coords = self.coordinates
        spatial = self.depth_embed(coords[:, 0]) + self.height_embed(coords[:, 1]) + self.width_embed(coords[:, 2])
        tokens = tokens + spatial[None, None, :, :] + self.roi_embed.weight[None, :, None, :]
        occupancy = F.max_pool3d((x.abs() > 1e-6).float(), kernel_size=self.patch_size, stride=self.patch_size)
        valid = occupancy.flatten(1).bool().reshape(batch_size, n_rois, self.patches_per_roi)
        tokens = tokens.reshape(batch_size, -1, self.d_model)
        valid = valid.reshape(batch_size, -1)
        tokens = tokens * valid.unsqueeze(-1).to(tokens.dtype)
        return tokens, valid


class CrossModalVisionMambaModel(nn.Module):
    """MRI and PET tokens attend to each other before pooling, then
    concatenated features are classified."""
    def __init__(self, n_rois=6, roi_size=64, patch_size=8, d_model=32, n_layers=2, n_classes=2, d_state=16, dropout=0.4):
        super().__init__()
        self.n_rois = n_rois
        self.patches_per_roi = (roi_size // patch_size) ** 3
        self.mri_patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.pet_patch_embed = ROIPatchEmbed3D(n_rois, roi_size, patch_size, d_model)
        self.mri_vim = VimEncoder(d_model, n_layers, d_state)
        self.pet_vim = VimEncoder(d_model, n_layers, d_state)
        self.cross_attn_mri_to_pet = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.cross_attn_pet_to_mri = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model * 2, n_classes)

    def forward(self, mri_rois, pet_rois, return_attention=False):
        mri_tokens, mri_valid = self.mri_patch_embed(mri_rois)
        pet_tokens, pet_valid = self.pet_patch_embed(pet_rois)

        mri_tokens = self.mri_vim(mri_tokens)
        pet_tokens = self.pet_vim(pet_tokens)

        # need_weights=True (default) returns averaged-over-heads attention weights
        # shape: (batch, query_len, key_len) -- e.g. (B, 3072, 3072)
        mri_attended, mri_attn = self.cross_attn_mri_to_pet(mri_tokens, pet_tokens, pet_tokens)
        pet_attended, pet_attn = self.cross_attn_pet_to_mri(pet_tokens, mri_tokens, mri_tokens)

        w_mri = mri_valid.unsqueeze(-1).to(mri_attended.dtype)
        mri_pooled = (mri_attended * w_mri).sum(dim=1) / w_mri.sum(dim=1).clamp_min(1.0)

        w_pet = pet_valid.unsqueeze(-1).to(pet_attended.dtype)
        pet_pooled = (pet_attended * w_pet).sum(dim=1) / w_pet.sum(dim=1).clamp_min(1.0)

        fused = torch.cat([mri_pooled, pet_pooled], dim=1)
        logits = self.classifier(self.dropout(fused))

        if return_attention:
            return logits, mri_attn, pet_attn
        return logits

In [4]:
COHORT_CSV     = "D:/mamba_model/thesis_cohort_final.csv"
MRI_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_roi64_aug"
PET_CACHE_AUG  = "D:/mamba_model/preprocessed_cache_pet_aug"
CKPT_DIR       = "D:/mamba_model/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(COHORT_CSV)
sessions, labels = df["mri_session"].values, df["outcome_label"].values
X_tv, X_test, y_tv, y_test = train_test_split(sessions, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)
session_to_subject = dict(zip(df["mri_session"], df["subject_id"]))

ROI_NAMES = ['Left-Hippocampus', 'Right-Hippocampus', 'Left-Cerebellum-WM',
             'Right-Cerebellum-WM', 'Left-Cerebral-WM', 'Right-Cerebral-WM']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 126 | Val: 42 | Test: 42


In [5]:
class MultimodalROIDataset(Dataset):
    def __init__(self, sessions, labels, mri_cache_dir, pet_cache_dir, is_train=False):
        self.samples = []
        self.mri_cache_dir, self.pet_cache_dir = mri_cache_dir, pet_cache_dir
        for session_id, label in zip(sessions, labels):
            subject_id = session_to_subject[session_id]
            self.samples.append((session_id, subject_id, label, "orig"))
            if is_train:
                for seed in [1, 101, 42]:
                    self.samples.append((session_id, subject_id, label, f"aug{seed}"))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        mri_key, pet_key, label, version = self.samples[idx]
        mri_rois = np.array(np.load(f"{self.mri_cache_dir}/{mri_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        pet_rois = np.array(np.load(f"{self.pet_cache_dir}/{pet_key}_{version}.npy", mmap_mode="r"), dtype=np.float32, copy=True)
        return torch.from_numpy(mri_rois).unsqueeze(1), torch.from_numpy(pet_rois).unsqueeze(1), torch.tensor(label, dtype=torch.long), mri_key

In [6]:
def train_epoch_mm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for mri_rois, pet_rois, labels, _ in loader:
        mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(mri_rois, pet_rois), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_mm(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for mri_rois, pet_rois, labels, _ in loader:
            mri_rois, pet_rois, labels = mri_rois.to(device), pet_rois.to(device), labels.to(device)
            out = model(mri_rois, pet_rois)
            total_loss += criterion(out, labels).item()
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    acc = np.mean(np.array(preds_all) == np.array(labels_all))
    tpr = recall_score(labels_all, preds_all, zero_division=0)
    tnr = specificity_score(labels_all, preds_all)
    return total_loss / len(loader), acc, tpr, tnr

In [7]:
def measure_inference_time(model, loader, device, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            mri_rois, pet_rois, labels, _ = batch
            mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
            bs = mri_rois.shape[0]
            if device.type == 'cuda': torch.cuda.synchronize()
            t0 = time.time()
            _ = model(mri_rois, pet_rois)
            if device.type == 'cuda': torch.cuda.synchronize()
            times.append((time.time() - t0) / bs)
    return np.mean(times), np.std(times)

def try_compute_flops(model, loader, device):
    try:
        model.eval()
        batch = next(iter(loader))
        with torch.no_grad():
            mri_rois, pet_rois, labels, _ = batch
            inputs = (mri_rois[:1].to(device), pet_rois[:1].to(device))
            macs, _ = profile(model, inputs=inputs, verbose=False)
        return macs * 2
    except Exception as e:
        print(f"  (FLOPs failed: {e})")
        return None

def run_one_seed(seed, train_loader, val_loader, test_loader, save_prefix, max_epochs=101, patience=15):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    model = CrossModalVisionMambaModel(d_model=32, n_layers=2, n_classes=2, dropout=0.4).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    best_val_loss, no_improve, best_epoch, total_time = float("inf"), 0, 0, 0
    save_path = f"{CKPT_DIR}/{save_prefix}_seed{seed}.pt"

    print(f"\n--- Seed {seed} ---")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Val Acc':>8} | {'Val TPR':>8} | {'Val TNR':>8} | {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, max_epochs):
        t0 = time.time()
        train_loss = train_epoch_mm(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_tpr, val_tnr = evaluate_mm(model, val_loader, criterion, device)
        scheduler.step(val_loss)
        epoch_time = time.time() - t0
        total_time += epoch_time
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | {val_acc:>8.4f} | {val_tpr:>8.4f} | {val_tnr:>8.4f} | {epoch_time:>5.1f}s")

        if val_loss < best_val_loss:
            best_val_loss, best_epoch, no_improve = val_loss, epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}. Best: {best_epoch}")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    test_loss, test_acc, test_tpr, test_tnr = evaluate_mm(model, test_loader, criterion, device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    inf_mean, inf_std = measure_inference_time(model, test_loader, device)
    flops = try_compute_flops(model, test_loader, device)

    print(f"\n  >>> Seed {seed} TEST: Acc={test_acc*100:.1f}% | TPR={test_tpr*100:.1f}% | TNR={test_tnr*100:.1f}% | "
          f"train_time={total_time/60:.1f}min | inf={inf_mean*1000:.2f}ms | {f'{flops/1e9:.2f}GFLOPs' if flops else 'N/A'}")

    return {"seed": seed, "acc": test_acc, "tpr": test_tpr, "tnr": test_tnr, "best_epoch": best_epoch,
            "train_time_sec": total_time, "n_params": n_params, "inf_time_ms": inf_mean * 1000, "flops": flops,
            "model_path": save_path}

In [8]:
BATCH_SIZE = 4
mm_train_loader = DataLoader(MultimodalROIDataset(X_train, y_train, MRI_CACHE_AUG, PET_CACHE_AUG, True), batch_size=BATCH_SIZE, shuffle=True)
mm_val_loader   = DataLoader(MultimodalROIDataset(X_val, y_val, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)
mm_test_loader  = DataLoader(MultimodalROIDataset(X_test, y_test, MRI_CACHE_AUG, PET_CACHE_AUG, False), batch_size=BATCH_SIZE, shuffle=False)

print("=== Cross-modal attention multimodal: seed 1 ===")
cross_attn_result = run_one_seed(1, mm_train_loader, mm_val_loader, mm_test_loader, "vim_crossattn_mm")

print(f"\nCompare to concatenation (no attention): Acc=65.1±1.4% | TPR=58.7±5.5% | TNR=71.4±4.8%")

=== Cross-modal attention multimodal: seed 1 ===

--- Seed 1 ---
 Epoch | Train Loss |   Val Loss |  Val Acc |  Val TPR |  Val TNR |   Time
----------------------------------------------------------------------
     1 |     0.6949 |     0.6946 |   0.5000 |   1.0000 |   0.0000 | 104.9s
     2 |     0.6932 |     0.6949 |   0.5000 |   1.0000 |   0.0000 |  49.4s
     3 |     0.6941 |     0.6922 |   0.5000 |   1.0000 |   0.0000 |  49.4s
     4 |     0.6907 |     0.6910 |   0.5000 |   1.0000 |   0.0000 |  49.3s
     5 |     0.6910 |     0.6907 |   0.5000 |   1.0000 |   0.0000 |  49.4s
     6 |     0.6898 |     0.6894 |   0.5000 |   1.0000 |   0.0000 |  49.4s
     7 |     0.6891 |     0.6900 |   0.5000 |   1.0000 |   0.0000 |  49.2s
     8 |     0.6887 |     0.6875 |   0.5476 |   0.9524 |   0.1429 |  49.2s
     9 |     0.6878 |     0.6863 |   0.5000 |   0.8095 |   0.1905 |  49.3s
    10 |     0.6823 |     0.6865 |   0.5000 |   1.0000 |   0.0000 |  49.3s
    11 |     0.6837 |     0.6822 |   0.

In [14]:
def extract_region_attention(model, loader, device, n_rois=6, patches_per_roi=512):
    model.eval()
    mri_region_scores = []
    pet_region_scores = []
    subject_keys = []

    with torch.no_grad():
        for mri_rois, pet_rois, labels, keys in loader:
            mri_rois, pet_rois = mri_rois.to(device), pet_rois.to(device)
            logits, mri_attn, pet_attn = model(mri_rois, pet_rois, return_attention=True)

            # Average over the QUERY dimension (dim=1), not keys (dim=2) --
            # this gives "how much attention did this key token RECEIVE
            # on average from all queries", which is not fixed by softmax's
            # row-sum-to-1 property and genuinely reflects learned importance.
            mri_received = mri_attn.mean(dim=1)  # (batch, 3072) -- attention each MRI key token received
            pet_received = pet_attn.mean(dim=1)

            batch_size = mri_received.shape[0]
            mri_received = mri_received.reshape(batch_size, n_rois, patches_per_roi).mean(dim=2)
            pet_received = pet_received.reshape(batch_size, n_rois, patches_per_roi).mean(dim=2)

            mri_region_scores.append(mri_received.cpu().numpy())
            pet_region_scores.append(pet_received.cpu().numpy())
            subject_keys.extend(keys)

    mri_region_scores = np.concatenate(mri_region_scores, axis=0)
    pet_region_scores = np.concatenate(pet_region_scores, axis=0)
    return mri_region_scores, pet_region_scores, subject_keys

In [22]:
ri_scores, pet_scores, keys = extract_region_attention(best_model, mm_test_loader, device)
# Attention values compare to the "no preference" baseline of 1/3072 = 0.000326.

print(f"{'Region':<22} | {'MRI attn mean':>16} | {'MRI attn std':>16}")
for i, roi_name in enumerate(ROI_NAMES):
    print(f"{roi_name:<22} | {mri_scores[:, i].mean():>16.8f} | {mri_scores[:, i].std():>16.8f}")

print("\nCompare to individual paired-ROI accuracy ranking (no stats, standard padding):")
print("  Cerebral-WM: 69.0% | Cerebellum-WM: 65.1% | Hippocampus: 56.3%")

Region                 |    MRI attn mean |     MRI attn std
Left-Hippocampus       |       0.00031409 |       0.00000549
Right-Hippocampus      |       0.00031449 |       0.00000560
Left-Cerebellum-WM     |       0.00031846 |       0.00000431
Right-Cerebellum-WM    |       0.00031890 |       0.00000475
Left-Cerebral-WM       |       0.00034416 |       0.00000632
Right-Cerebral-WM      |       0.00034304 |       0.00000772

Compare to individual paired-ROI accuracy ranking (no stats, standard padding):
  Cerebral-WM: 69.0% | Cerebellum-WM: 65.1% | Hippocampus: 56.3%


In [21]:
# Sanity check: one query's attention weights should sum to ~1.0
sample_query_weights = mri_attn_sample[0, 0, :]  # first subject, first query token, all 3072 keys
print(f"Sum of one query's attention weights: {sample_query_weights.sum().item():.6f}")

Sum of one query's attention weights: 1.000000


In [23]:
print("=== Region attention ranking (highest to lowest, MRI) ===")
ranking = sorted(zip(ROI_NAMES, mri_scores.mean(axis=0)), key=lambda x: -x[1])
for name, score in ranking:
    print(f"  {name}: {score:.5f}")

=== Region attention ranking (highest to lowest, MRI) ===
  Left-Cerebral-WM: 0.00034
  Right-Cerebral-WM: 0.00034
  Right-Cerebellum-WM: 0.00032
  Left-Cerebellum-WM: 0.00032
  Right-Hippocampus: 0.00031
  Left-Hippocampus: 0.00031


In [24]:
baseline = 1 / 3072  # 0.000326 -- what every region would get with zero preference

print(f"{'Region':<22} | {'Mean attn':>12} | {'vs. baseline':>14}")
print("-" * 55)
for i, roi_name in enumerate(ROI_NAMES):
    mean_val = mri_scores[:, i].mean()
    pct_vs_baseline = ((mean_val - baseline) / baseline) * 100
    sign = "+" if pct_vs_baseline >= 0 else ""
    print(f"{roi_name:<22} | {mean_val:>12.6f} | {sign}{pct_vs_baseline:>6.1f}%")

Region                 |    Mean attn |   vs. baseline
-------------------------------------------------------
Left-Hippocampus       |     0.000314 |   -3.5%
Right-Hippocampus      |     0.000314 |   -3.4%
Left-Cerebellum-WM     |     0.000318 |   -2.2%
Right-Cerebellum-WM    |     0.000319 |   -2.0%
Left-Cerebral-WM       |     0.000344 | +   5.7%
Right-Cerebral-WM      |     0.000343 | +   5.4%
